# 🌳 Rainforest Deforestation Monitoring and Prediction Using Machine Learning

**Monitor Deforestation Rates in Rainforests Using Time-Series Remote Sensing for Conservation Efforts**

> Run all cells top-to-bottom in Jupyter or Google Colab. No Streamlit or external server needed.

---
## Contents
1. [Setup](#setup)
2. [Data Loading & Inspection](#loading)
3. [Preprocessing](#preprocessing)
4. [Exploratory Data Analysis](#eda)
5. [Feature Engineering](#features)
6. [Model Training](#training)
7. [Model Evaluation](#evaluation)
8. [Future Forecasting (2023–2027)](#forecasting)
9. [Risk Assessment](#risk)
10. [Carbon Emissions Analysis](#carbon)
11. [Summary & Conclusions](#summary)


---
<a id="setup"></a>
## 1 · Setup & Configuration


In [1]:
# ── Google Colab only: uncomment the two lines below ──
# from google.colab import drive
# drive.mount('/content/drive')

import os, sys, warnings
warnings.filterwarnings('ignore')

# Auto-detect project root (works when notebook is in the project folder).
# For Colab, set the path explicitly:
# PROJECT_DIR = "/content/drive/MyDrive/Rainforest Deforestation Monitoring and Prediction Using Machine Learning"
PROJECT_DIR = os.path.abspath('.')
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")


Working directory: d:\Project\Rainforest Deforestation Monitoring and Prediction Using Machine Learning


In [2]:
# Install deps (uncomment if running for the first time)
# !pip install -q pandas numpy matplotlib plotly scikit-learn xgboost statsmodels joblib openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML

import config
from src.data_loader import load_all_datasets
from src.preprocessing import (
    preprocess_subnational_tree_loss, preprocess_carbon_data,
    merge_carbon_with_loss, extract_annual_columns,
)
from src.feature_engineering import build_feature_set, get_feature_columns, FEATURE_DOCUMENTATION
from src.train import train_all_models, chronological_split, predict_with_model
from src.evaluate import evaluate_all_models, evaluate_by_region, build_prediction_table
from src.forecasting import forecast_region, compute_forecast_uncertainty, apply_uncertainty_intervals
from src.risk_assessment import full_risk_assessment
from src.eda import summary_statistics, compute_regional_trends, correlation_analysis

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print("All imports successful ✅")


All imports successful ✅


---
<a id="loading"></a>
## 2 · Data Loading & Inspection

Loading all 4 Global Forest Change CSV datasets (Hansen et al., 2013).


In [3]:
datasets = load_all_datasets()

names = {
    'country_loss': 'Country Tree Cover Loss',
    'subnational_loss': 'Subnational Tree Cover Loss',
    'country_carbon': 'Country Carbon Data',
    'subnational_carbon': 'Subnational Carbon Data',
}
print("=" * 70)
print("DATASET INVENTORY")
print("=" * 70)
for key, name in names.items():
    df = datasets.get(key)
    if df is not None:
        print(f"\n📄 {name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    else:
        print(f"\n❌ {name}: NOT FOUND")


2026-11-21 20:44:06,361 - deforestation_ml.data_loader - INFO - Loaded Country Tree Cover Loss: 1888 rows × 28 columns from Country tree cover loss.csv
2026-11-21 20:44:06,475 - deforestation_ml.data_loader - INFO - Loaded Subnational Tree Cover Loss: 28000 rows × 29 columns from Subnational 1 tree cover loss.csv
2026-11-21 20:44:06,594 - deforestation_ml.data_loader - INFO - Loaded Country Carbon Data: 1888 rows × 30 columns from Country carbon data.csv
2026-11-21 20:44:06,733 - deforestation_ml.data_loader - INFO - Loaded Subnational Carbon Data: 28000 rows × 31 columns from Subnational 1 carbon data.csv


DATASET INVENTORY

📄 Country Tree Cover Loss: 1,888 rows × 28 cols

📄 Subnational Tree Cover Loss: 28,000 rows × 29 cols

📄 Country Carbon Data: 1,888 rows × 30 cols

📄 Subnational Carbon Data: 28,000 rows × 31 cols


In [4]:
# Primary dataset — Subnational Tree Cover Loss
sub = datasets['subnational_loss']
print("PRIMARY DATASET: Subnational Tree Cover Loss")
print(f"Shape: {sub.shape}\n")
print("Data types:")
print(sub.dtypes.to_string())
print(f"\nMissing values: {sub.isnull().sum().sum()}")
print(f"\nSample rows:")
display(sub.head(3))


PRIMARY DATASET: Subnational Tree Cover Loss
Shape: (28000, 29)

Data types:
country                str
subnational1           str
threshold            int64
area_ha              int64
extent_2000_ha       int64
extent_2010_ha       int64
gain_2000-2020_ha    int64
tc_loss_ha_2001      int64
tc_loss_ha_2002      int64
tc_loss_ha_2003      int64
tc_loss_ha_2004      int64
tc_loss_ha_2005      int64
tc_loss_ha_2006      int64
tc_loss_ha_2007      int64
tc_loss_ha_2008      int64
tc_loss_ha_2009      int64
tc_loss_ha_2010      int64
tc_loss_ha_2011      int64
tc_loss_ha_2012      int64
tc_loss_ha_2013      int64
tc_loss_ha_2014      int64
tc_loss_ha_2015      int64
tc_loss_ha_2016      int64
tc_loss_ha_2017      int64
tc_loss_ha_2018      int64
tc_loss_ha_2019      int64
tc_loss_ha_2020      int64
tc_loss_ha_2021      int64
tc_loss_ha_2022      int64

Missing values: 0

Sample rows:


,country,subnational1,threshold,area_ha,extent_2000_ha,extent_2010_ha,gain_2000-2020_ha,tc_loss_ha_2001,tc_loss_ha_2002,tc_loss_ha_2003,tc_loss_ha_2004,tc_loss_ha_2005,tc_loss_ha_2006,tc_loss_ha_2007,tc_loss_ha_2008,tc_loss_ha_2009,tc_loss_ha_2010,tc_loss_ha_2011,tc_loss_ha_2012,tc_loss_ha_2013,tc_loss_ha_2014,tc_loss_ha_2015,tc_loss_ha_2016,tc_loss_ha_2017,tc_loss_ha_2018,tc_loss_ha_2019,tc_loss_ha_2020,tc_loss_ha_2021,tc_loss_ha_2022
0,Afghanistan,Badakhshan,0,4369203,4369203,4369203,403,3,9,1,6,5,3,2,3,6,3,1,12,2,4,0,0,0,0,0,0,0,0
1,Afghanistan,Badakhshan,10,4369203,10553,1763,403,0,0,0,0,0,0,0,1,1,0,0,1,0,1,0,0,0,0,0,0,0,0
2,Afghanistan,Badakhshan,15,4369203,3074,914,403,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0


In [5]:
# Geographic coverage
brazil = sub[sub['country'] == 'Brazil']
regions = sorted(brazil['subnational1'].unique())
print(f"Countries: {sub['country'].nunique()}")
print(f"Brazil subnational regions: {len(regions)}\n")
for i, r in enumerate(regions, 1):
    print(f"  {i:2d}. {r}")


Countries: 219
Brazil subnational regions: 27

   1. Acre
   2. Alagoas
   3. Amapá
   4. Amazonas
   5. Bahia
   6. Ceará
   7. Distrito Federal
   8. Espírito Santo
   9. Goiás
  10. Maranhão
  11. Mato Grosso
  12. Mato Grosso do Sul
  13. Minas Gerais
  14. Paraná
  15. Paraíba
  16. Pará
  17. Pernambuco
  18. Piauí
  19. Rio Grande do Norte
  20. Rio Grande do Sul
  21. Rio de Janeiro
  22. Rondônia
  23. Roraima
  24. Santa Catarina
  25. Sergipe
  26. São Paulo
  27. Tocantins


---
<a id="preprocessing"></a>
## 3 · Data Preprocessing

**Steps:** Filter to threshold ≥30 % → Brazil → melt 22 annual columns to long format → calculate loss rate → merge carbon data.

> **Important:** Tree-cover loss ≠ permanent deforestation. It includes fire, plantation harvest, and natural disturbances.


In [6]:
processed = preprocess_subnational_tree_loss(
    datasets['subnational_loss'], threshold=30, country='Brazil'
)
carbon_long = preprocess_carbon_data(
    datasets['subnational_carbon'], threshold=30, country='Brazil'
)
processed = merge_carbon_with_loss(processed, carbon_long)

print("PROCESSED DATASET")
print("=" * 50)
print(f"Shape: {processed.shape}")
print(f"Regions: {processed['region'].nunique()}")
print(f"Years: {processed['year'].min()}–{processed['year'].max()}")
print(f"Missing values: {processed.isnull().sum().sum()}")
print(f"\nColumns: {list(processed.columns)}\n")
display(processed.head())


2026-11-21 20:44:07,029 - deforestation_ml.preprocessing - INFO - Filtered to threshold=30: 3500 rows (from 28000)
2026-11-21 20:44:07,039 - deforestation_ml.preprocessing - INFO - Filtered to country='Brazil': 27 rows


2026-11-21 20:44:07,145 - deforestation_ml.preprocessing - INFO - Melted 22 annual columns → 594 rows, years 2001–2022
2026-11-21 20:44:07,210 - deforestation_ml.preprocessing - INFO - Preprocessing complete: 594 rows, 27 regions, years 2001–2022
2026-11-21 20:44:07,230 - deforestation_ml.preprocessing - INFO - Filtered to threshold=30: 3500 rows (from 28000)
2026-11-21 20:44:07,327 - deforestation_ml.preprocessing - INFO - Merged carbon data: 594/594 rows matched (100.0%)


PROCESSED DATASET
Shape: (594, 12)
Regions: 27
Years: 2001–2022
Missing values: 0

Columns: ['country', 'region', 'total_area_ha', 'extent_2000_ha', 'extent_2010_ha', 'gain_2000_2020_ha', 'tc_loss_ha', 'year', 'loss_rate_pct', 'cumulative_loss_ha', 'cumulative_loss_pct', 'carbon_emissions_Mg_CO2e']



,country,region,total_area_ha,extent_2000_ha,extent_2010_ha,gain_2000_2020_ha,tc_loss_ha,year,loss_rate_pct,cumulative_loss_ha,cumulative_loss_pct,carbon_emissions_Mg_CO2e
0,Brazil,Acre,15273530,14379380,14079300,66320,67310,2001,0.47,67310,0.47,"43,237,142.00"
1,Brazil,Acre,15273530,14379380,14079300,66320,68439,2002,0.48,135749,0.94,"48,986,317.00"
2,Brazil,Acre,15273530,14379380,14079300,66320,37666,2003,0.26,173415,1.21,"26,082,167.00"
3,Brazil,Acre,15273530,14379380,14079300,66320,67190,2004,0.47,240605,1.67,"48,123,664.00"
4,Brazil,Acre,15273530,14379380,14079300,66320,169860,2005,1.18,410465,2.85,"117,949,735.00"


### Data Dictionary

| Column | Unit | Description |
|--------|------|-------------|
| `region` | — | Subnational region name |
| `country` | — | Country (Brazil) |
| `year` | — | Observation year (2001–2022) |
| `tc_loss_ha` | hectares | Annual tree-cover loss |
| `extent_2000_ha` | hectares | Baseline tree-cover extent (year 2000) |
| `loss_rate_pct` | % | `(tc_loss_ha / extent_2000_ha) × 100` |
| `cumulative_loss_ha` | hectares | Cumulative loss since 2001 |
| `cumulative_loss_pct` | % | Cumulative loss as % of baseline |
| `carbon_emissions_Mg_CO2e` | Mg CO₂e | Gross carbon emissions from GFW model |


---
<a id="eda"></a>
## 4 · Exploratory Data Analysis


### 4.1  Key Metrics


In [7]:
total_loss = processed['tc_loss_ha'].sum()
annual_totals = processed.groupby('year')['tc_loss_ha'].sum()
peak_year = annual_totals.idxmax()
peak_val = annual_totals.max()

print("KEY METRICS")
print("=" * 50)
print(f"Total tree-cover loss (2001–2022):  {total_loss:,.0f} ha  ({total_loss/1e6:.1f} M ha)")
print(f"Peak year:                          {peak_year}  ({peak_val:,.0f} ha)")
print(f"Average annual loss:                {total_loss/22:,.0f} ha/year")
if 'carbon_emissions_Mg_CO2e' in processed.columns:
    tc = processed['carbon_emissions_Mg_CO2e'].sum()
    print(f"Total carbon emissions:             {tc:,.0f} Mg CO₂e  ({tc/1e9:.1f} B Mg)")


KEY METRICS
Total tree-cover loss (2001–2022):  66,126,533 ha  (66.1 M ha)
Peak year:                          2016  (5,378,844 ha)
Average annual loss:                3,005,752 ha/year
Total carbon emissions:             36,134,302,097 Mg CO₂e  (36.1 B Mg)


### 4.2  Total Annual Tree-Cover Loss


In [8]:
annual = processed.groupby('year')['tc_loss_ha'].sum().reset_index()
fig = px.bar(annual, x='year', y='tc_loss_ha',
             title='Total Annual Tree-Cover Loss in Brazil (2001–2022)',
             labels={'tc_loss_ha': 'Tree-Cover Loss (hectares)', 'year': 'Year'},
             color='tc_loss_ha', color_continuous_scale='YlOrRd')
fig.update_layout(template='plotly_white', showlegend=False, height=450)
fig.show()


### 4.3  Regional Comparison


In [9]:
reg = processed.groupby('region')['tc_loss_ha'].sum().sort_values(ascending=True).reset_index()
fig = px.bar(reg, x='tc_loss_ha', y='region', orientation='h',
             title='Total Tree-Cover Loss by Region (2001-2022)',
             labels={'tc_loss_ha': 'Total Loss (ha)', 'region': ''},
             color='tc_loss_ha', color_continuous_scale='YlOrRd')
fig.update_layout(template='plotly_white', height=700, showlegend=False)
fig.show()


### 4.4  Top 10 Regions — Annual Trends


In [10]:
top10 = processed.groupby('region')['tc_loss_ha'].sum().nlargest(10).index
fig = px.line(processed[processed['region'].isin(top10)],
              x='year', y='tc_loss_ha', color='region',
              title='Annual Tree-Cover Loss — Top 10 Regions',
              labels={'tc_loss_ha': 'Loss (ha)', 'year': 'Year'}, markers=True)
fig.update_layout(template='plotly_white', height=500)
fig.show()


### 4.5  Heatmap — Region × Year


In [11]:
pivot = processed.pivot(index='region', columns='year', values='tc_loss_ha')
fig = px.imshow(pivot, aspect='auto',
                title='Tree-Cover Loss Heatmap (Region × Year)',
                labels={'color': 'Loss (ha)'}, color_continuous_scale='YlOrRd')
fig.update_layout(height=700)
fig.show()


### 4.6  Loss-Rate Heatmap


In [12]:
pivot_rate = processed.pivot(index='region', columns='year', values='loss_rate_pct')
fig = px.imshow(pivot_rate, aspect='auto',
                title='Annual Forest-Loss Rate (%) — Region × Year',
                labels={'color': 'Rate (%)'}, color_continuous_scale='RdYlGn_r')
fig.update_layout(height=700)
fig.show()


### 4.7  Cumulative Loss — Top 5


In [13]:
top5 = processed.groupby('region')['tc_loss_ha'].sum().nlargest(5).index
fig = px.line(processed[processed['region'].isin(top5)],
              x='year', y='cumulative_loss_ha', color='region',
              title='Cumulative Tree-Cover Loss — Top 5 Regions',
              labels={'cumulative_loss_ha': 'Cumulative Loss (ha)', 'year': 'Year'},
              markers=True)
fig.update_layout(template='plotly_white', height=450)
fig.show()


### 4.8  Trend Analysis


In [14]:
trends = compute_regional_trends(processed)
inc = trends[trends['trend_direction'] == 'increasing'].sort_values('trend_slope', ascending=False)
dec = trends[trends['trend_direction'] == 'decreasing'].sort_values('trend_slope')
print(f"Regions with INCREASING loss trends: {len(inc)}")
display(inc[['region', 'trend_slope', 'total_loss']].head(10))
print(f"\nRegions with DECREASING loss trends: {len(dec)}")
display(dec[['region', 'trend_slope', 'total_loss']].head(10))


Regions with INCREASING loss trends: 0


,region,trend_slope,total_loss



Regions with DECREASING loss trends: 0


,region,trend_slope,total_loss


### 4.9  Correlation Matrix


In [15]:
num_cols = ['tc_loss_ha', 'extent_2000_ha', 'loss_rate_pct']
if 'carbon_emissions_Mg_CO2e' in processed.columns:
    num_cols.append('carbon_emissions_Mg_CO2e')
corr = processed[num_cols].corr()
fig = px.imshow(corr, text_auto='.3f', aspect='auto',
                title='Correlation Matrix', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.update_layout(height=450, width=550)
fig.show()


### 4.10  Regional Rankings


In [16]:
rankings = processed.groupby('region').agg(
    total_loss_ha=('tc_loss_ha', 'sum'),
    avg_annual_loss=('tc_loss_ha', 'mean'),
    avg_loss_rate=('loss_rate_pct', 'mean'),
).sort_values('total_loss_ha', ascending=False)
print("TOP 10 REGIONS BY TOTAL LOSS")
display(rankings.head(10))
print("\nBOTTOM 5 REGIONS BY TOTAL LOSS")
display(rankings.tail(5))


TOP 10 REGIONS BY TOTAL LOSS


,total_loss_ha,avg_annual_loss,avg_loss_rate
region,,,
Pará,16330101,"742,277.32",0.68
Mato Grosso,13171668,"598,712.18",1.05
Maranhão,5881361,"267,334.59",1.27
Rondônia,4948995,"224,954.32",1.20
Amazonas,4020611,"182,755.05",0.12
Bahia,3638717,"165,396.23",0.87
Minas Gerais,3002582,"136,481.00",0.74
Tocantins,2159950,"98,179.55",0.87
Mato Grosso do Sul,1729526,"78,614.82",0.76



BOTTOM 5 REGIONS BY TOTAL LOSS


,total_loss_ha,avg_annual_loss,avg_loss_rate
region,,,
Paraíba,128896,"5,858.91",0.52
Rio Grande do Norte,115195,"5,236.14",0.57
Rio de Janeiro,80645,"3,665.68",0.20
Alagoas,77463,"3,521.05",0.60
Distrito Federal,3271,148.68,0.26


---
<a id="features"></a>
## 5 · Feature Engineering

15 leakage-safe time-series features per region. All use `shift(1)` or earlier — no future data leaks.


In [17]:
featured = build_feature_set(processed)
feature_cols = get_feature_columns(featured)
print(f"Dataset shape: {featured.shape}")
print(f"Features: {len(feature_cols)}\n")
for i, c in enumerate(feature_cols, 1):
    print(f"  {i:2d}. {c}")
print(f"\nRows with NaN features (early years, expected): {featured[feature_cols].isnull().any(axis=1).sum()}")


2026-11-21 20:44:10,004 - deforestation_ml.feature_engineering - INFO - Starting feature engineering...
2026-11-21 20:44:10,125 - deforestation_ml.feature_engineering - INFO - Encoded 27 regions
2026-11-21 20:44:10,128 - deforestation_ml.feature_engineering - INFO - Feature engineering complete: 15 features created


Dataset shape: (594, 27)
Features: 15

   1. tc_loss_ha_lag1
   2. tc_loss_ha_lag2
   3. tc_loss_ha_lag3
   4. tc_loss_ha_roll_mean_3yr
   5. tc_loss_ha_roll_std_3yr
   6. tc_loss_ha_roll_mean_5yr
   7. tc_loss_ha_roll_std_5yr
   8. loss_change_1yr_lagged
   9. loss_change_pct_1yr_lagged
  10. loss_acceleration
  11. loss_trend_3yr
  12. loss_rate_pct_lag1
  13. cumulative_loss_pct_lag1
  14. loss_cv_5yr
  15. region_encoded

Rows with NaN features (early years, expected): 108


In [18]:
print(FEATURE_DOCUMENTATION)



Feature Documentation

Target Variable:
  tc_loss_ha — Annual tree-cover loss in hectares for the given region and year.

Lag Features:
  tc_loss_ha_lag1 — Tree-cover loss in the previous year.
    Rationale: Most direct predictor; deforestation tends to persist year-over-year.
    
  tc_loss_ha_lag2 — Tree-cover loss two years prior.
    Rationale: Captures medium-term persistence.
    
  tc_loss_ha_lag3 — Tree-cover loss three years prior.
    Rationale: Captures longer-term baseline deforestation level.

Rolling Features:
  tc_loss_ha_roll_mean_3yr — Mean loss over the prior 3 years.
    Rationale: Smoothed estimate of recent deforestation activity.
    
  tc_loss_ha_roll_std_3yr — Standard deviation of loss over the prior 3 years.
    Rationale: Captures volatility in recent deforestation.
    
  tc_loss_ha_roll_mean_5yr — Mean loss over the prior 5 years.
    Rationale: Longer-term average deforestation level.
    
  tc_loss_ha_roll_std_5yr — Standard deviation of loss over the p

In [19]:
fig = make_subplots(rows=3, cols=3, subplot_titles=feature_cols[:9])
for i, col in enumerate(feature_cols[:9]):
    r, c = divmod(i, 3)
    fig.add_trace(go.Histogram(x=featured[col].dropna(), name=col, showlegend=False), row=r+1, col=c+1)
fig.update_layout(title='Feature Distributions (first 9)', height=600, template='plotly_white')
fig.show()


---
<a id="training"></a>
## 6 · Model Training

| Model | Type | Description |
|-------|------|-------------|
| Historical Baseline | Naive | Previous year's loss as prediction |
| Linear Regression | Linear | Scaled features, interpretable |
| Random Forest | Ensemble | 200 trees, max depth 15 |
| XGBoost | Gradient Boosting | 200 estimators |

**Chronological split (no random shuffling):** Train ≤2017 · Val 2018-2019 · Test 2020-2022


In [20]:
splits = chronological_split(
    featured,
    train_end_year=config.TRAIN_END_YEAR,
    val_end_year=config.VAL_END_YEAR,
    feature_cols=feature_cols,
)

X_train, y_train = splits['train']
X_val,   y_val   = splits['val']
X_test,  y_test  = splits['test']

print("CHRONOLOGICAL SPLIT")
print("=" * 50)
print(f"Train: {len(X_train)} rows  (years ≤ {config.TRAIN_END_YEAR})")
print(f"Val:   {len(X_val)} rows  (years {config.TRAIN_END_YEAR+1}–{config.VAL_END_YEAR})")
print(f"Test:  {len(X_test)} rows  (years > {config.VAL_END_YEAR})")
print(f"\nFeatures: {len(feature_cols)}  |  Target: tc_loss_ha")


2026-11-21 20:44:10,287 - deforestation_ml.train - INFO - Chronological split — Train: 351 rows (≤2017), Val: 54 rows (2018–2019), Test: 81 rows (>2019)


CHRONOLOGICAL SPLIT
Train: 351 rows  (years ≤ 2017)
Val:   54 rows  (years 2018–2019)
Test:  81 rows  (years > 2019)

Features: 15  |  Target: tc_loss_ha


In [21]:
trained = train_all_models(splits)

print("\nTRAINED MODELS")
print("=" * 50)
for name, bundle in trained.items():
    print(f"  ✅ {name}: {type(bundle['model']).__name__}")


2026-11-21 20:44:10,309 - deforestation_ml.train - INFO - Training Model A: Historical Baseline...
2026-11-21 20:44:10,311 - deforestation_ml.train - INFO - Training Model B: Linear Regression...
2026-11-21 20:44:10,349 - deforestation_ml.train - INFO - Linear Regression trained. R² (train): 0.7903
2026-11-21 20:44:10,350 - deforestation_ml.train - INFO - Training Model C: Random Forest...
2026-11-21 20:44:10,754 - deforestation_ml.train - INFO - Random Forest trained. R² (train): 0.8641
2026-11-21 20:44:10,755 - deforestation_ml.train - INFO - Training Model D: XGBoost...
2026-11-21 20:44:12,437 - deforestation_ml.train - INFO - XGBoost trained. R² (train): 1.0000
2026-11-21 20:44:12,440 - deforestation_ml.train - INFO - Saved Historical Baseline to D:\Project\Rainforest Deforestation Monitoring and Prediction Using Machine Learning\models\historical_baseline_bundle.joblib
2026-11-21 20:44:12,446 - deforestation_ml.train - INFO - Saved Linear Regression to D:\Project\Rainforest Defore


TRAINED MODELS
  ✅ Historical Baseline: HistoricalBaseline
  ✅ Linear Regression: LinearRegression
  ✅ Random Forest: RandomForestRegressor
  ✅ XGBoost: XGBRegressor


---
<a id="evaluation"></a>
## 7 · Model Evaluation

All metrics from real predictions — nothing fabricated.


### 7.1  Comparison Tables


In [22]:
val_df, _   = evaluate_all_models(trained, splits, 'val')
test_df, _  = evaluate_all_models(trained, splits, 'test')

print("VALIDATION SET (2018–2019)")
print("=" * 60)
display(val_df)

print("\nTEST SET (2020–2022)")
print("=" * 60)
display(test_df)


2026-11-21 20:44:12,572 - deforestation_ml.evaluate - INFO - Historical Baseline — MAE: 40,808 ha, RMSE: 100,470 ha, R²: 0.6304 (n=54)
2026-11-21 20:44:12,582 - deforestation_ml.evaluate - INFO - Linear Regression — MAE: 77,739 ha, RMSE: 132,092 ha, R²: 0.3611 (n=54)
2026-11-21 20:44:12,653 - deforestation_ml.evaluate - INFO - Random Forest — MAE: 54,760 ha, RMSE: 104,482 ha, R²: 0.6003 (n=54)
2026-11-21 20:44:12,662 - deforestation_ml.evaluate - INFO - XGBoost — MAE: 65,120 ha, RMSE: 123,182 ha, R²: 0.4443 (n=54)
2026-11-21 20:44:12,671 - deforestation_ml.evaluate - INFO - Historical Baseline — MAE: 26,092 ha, RMSE: 69,295 ha, R²: 0.8697 (n=81)
2026-11-21 20:44:12,679 - deforestation_ml.evaluate - INFO - Linear Regression — MAE: 53,133 ha, RMSE: 101,483 ha, R²: 0.7205 (n=81)
2026-11-21 20:44:12,748 - deforestation_ml.evaluate - INFO - Random Forest — MAE: 37,386 ha, RMSE: 77,939 ha, R²: 0.8352 (n=81)
2026-11-21 20:44:12,758 - deforestation_ml.evaluate - INFO - XGBoost — MAE: 52,121 ha

VALIDATION SET (2018–2019)


,Model,MAE,RMSE,R2,n_samples
0,Historical Baseline,"40,807.80","100,469.79",0.63,54
1,Random Forest,"54,760.48","104,481.63",0.60,54
2,XGBoost,"65,119.85","123,181.97",0.44,54
3,Linear Regression,"77,738.84","132,092.36",0.36,54



TEST SET (2020–2022)


,Model,MAE,RMSE,R2,n_samples
0,Historical Baseline,"26,091.77","69,294.80",0.87,81
1,Random Forest,"37,385.88","77,939.03",0.84,81
2,XGBoost,"52,121.05","113,591.60",0.65,81
3,Linear Regression,"53,132.62","101,483.19",0.72,81


### 7.2  Comparison Chart


In [23]:
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=['MAE ↓', 'RMSE ↓', 'R² ↑'])
colors = px.colors.qualitative.Set2[:len(test_df)]
for i, (metric, ci) in enumerate([('MAE',1), ('RMSE',2), ('R2',3)]):
    fig.add_trace(go.Bar(x=test_df['Model'], y=test_df[metric],
                         marker_color=colors, showlegend=False), row=1, col=ci)
fig.update_layout(title='Model Comparison — Test Set', height=400, template='plotly_white')
fig.show()


### 7.3  Actual vs Predicted — Best Model


In [24]:
best_model_name = test_df.loc[test_df['MAE'].idxmin(), 'Model']
pred_table = build_prediction_table(trained, splits, 'test')

pred_col = f"pred_{best_model_name.lower().replace(' ', '_')}"
fig = px.scatter(pred_table, x='actual', y=pred_col,
                 hover_data=['region', 'year'],
                 title=f'Actual vs Predicted — {best_model_name} (Test Set)',
                 labels={'actual': 'Actual Loss (ha)', pred_col: 'Predicted Loss (ha)'})
mx = pred_table['actual'].max()
fig.add_shape(type='line', x0=0, y0=0, x1=mx, y1=mx,
              line=dict(dash='dash', color='red'))
fig.update_layout(template='plotly_white', height=500, width=600)
fig.show()


### 7.4  All Models — Actual vs Predicted


In [25]:
model_names = list(trained.keys())
fig = make_subplots(rows=2, cols=2, subplot_titles=model_names)
for i, name in enumerate(model_names):
    r, c = divmod(i, 2)
    col = f"pred_{name.lower().replace(' ', '_')}"
    fig.add_trace(go.Scatter(x=pred_table['actual'], y=pred_table[col],
                             mode='markers', name=name, showlegend=False), row=r+1, col=c+1)
    fig.add_shape(type='line', x0=0, y0=0, x1=mx, y1=mx,
                  line=dict(dash='dash', color='red'), row=r+1, col=c+1)
fig.update_layout(title='Actual vs Predicted — All Models', height=700, template='plotly_white')
fig.show()


### 7.5  Feature Importance (Random Forest)


In [26]:
rf_model = trained['Random Forest']['model']
if hasattr(rf_model, 'feature_importances_'):
    imp = pd.DataFrame({'feature': feature_cols, 'importance': rf_model.feature_importances_})
    imp = imp.sort_values('importance', ascending=True)
    fig = px.bar(imp, x='importance', y='feature', orientation='h',
                 title='Feature Importance — Random Forest',
                 labels={'importance': 'Importance', 'feature': ''})
    fig.update_layout(template='plotly_white', height=500)
    fig.show()
    print("⚠️  Feature importance = statistical association, NOT causation.")


⚠️  Feature importance = statistical association, NOT causation.


### 7.6  Per-Region Metrics


In [27]:
region_metrics = evaluate_by_region(trained, splits, 'test')
best_rm = region_metrics[region_metrics['Model'] == best_model_name].sort_values('MAE')
print(f"PER-REGION METRICS — {best_model_name}\n")
display(best_rm[['Region', 'MAE', 'RMSE', 'R2']].reset_index(drop=True))


2026-11-21 20:44:13,327 - deforestation_ml.evaluate - INFO - Historical Baseline — Acre — MAE: 18,786 ha, RMSE: 21,153 ha, R²: -0.0268 (n=3)
2026-11-21 20:44:13,333 - deforestation_ml.evaluate - INFO - Historical Baseline — Alagoas — MAE: 737 ha, RMSE: 757 ha, R²: -2.4790 (n=3)
2026-11-21 20:44:13,338 - deforestation_ml.evaluate - INFO - Historical Baseline — Amapá — MAE: 1,965 ha, RMSE: 2,026 ha, R²: -2.4410 (n=3)
2026-11-21 20:44:13,344 - deforestation_ml.evaluate - INFO - Historical Baseline — Amazonas — MAE: 60,074 ha, RMSE: 64,521 ha, R²: -0.0685 (n=3)
2026-11-21 20:44:13,349 - deforestation_ml.evaluate - INFO - Historical Baseline — Bahia — MAE: 21,285 ha, RMSE: 28,528 ha, R²: -0.2714 (n=3)
2026-11-21 20:44:13,353 - deforestation_ml.evaluate - INFO - Historical Baseline — Ceará — MAE: 429 ha, RMSE: 502 ha, R²: -1.2590 (n=3)
2026-11-21 20:44:13,359 - deforestation_ml.evaluate - INFO - Historical Baseline — Distrito Federal — MAE: 51 ha, RMSE: 56 ha, R²: -4.2272 (n=3)
2026-11-21 20

PER-REGION METRICS — Historical Baseline



,Region,MAE,RMSE,R2
0,Distrito Federal,51.33,55.66,-4.23
1,Ceará,428.67,501.70,-1.26
2,Paraíba,455.33,482.07,-0.27
3,Rio de Janeiro,555.33,582.03,-2.21
4,Alagoas,736.67,757.00,-2.48
5,Pernambuco,862.00,"1,012.32",-2.86
6,Sergipe,875.33,892.80,-4.48
7,Rio Grande do Norte,967.67,"1,151.28",-1.90
8,Espírito Santo,"1,492.67","1,763.75",-2.27
9,Santa Catarina,"1,798.00","2,889.84",-0.53


---
<a id="forecasting"></a>
## 8 · Future Forecasting (2023–2027)

Features for future years are generated **recursively** using prior predictions.
Uncertainty intervals come from test-set residuals (±1.96σ for 95 % CI).

> ⚠️ Forecasts are **estimates** — they cannot predict policy changes, fires, or economic shifts.


In [28]:
best_bundle = trained[best_model_name]
uncertainty = compute_forecast_uncertainty(trained, splits, model_name=best_model_name)
print(f"Best model: {best_model_name}")
print(f"Residual std (uncertainty): {uncertainty['residual_std']:,.0f} ha")

all_fc = []
for region in processed['region'].unique():
    fc = forecast_region(featured, region, best_bundle, horizon=5)
    fc = apply_uncertainty_intervals(fc, uncertainty)
    all_fc.append(fc)

forecasts_df = pd.concat(all_fc, ignore_index=True)
future_only = forecasts_df[forecasts_df['type'] == 'forecast']
print(f"\nForecasted {future_only['region'].nunique()} regions × {future_only['year'].nunique()} years = {len(future_only)} rows")


Best model: Historical Baseline
Residual std (uncertainty): 68,881 ha

Forecasted 27 regions × 5 years = 135 rows


### 8.1  Forecast Charts — Top 5 Regions


In [29]:
top5_fc = processed.groupby('region')['tc_loss_ha'].sum().nlargest(5).index
for region in top5_fc:
    rdf = forecasts_df[forecasts_df['region'] == region]
    hist = rdf[rdf['type'] == 'observed']
    fut  = rdf[rdf['type'] == 'forecast']

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist['year'], y=hist['predicted_loss_ha'],
                             mode='lines+markers', name='Historical',
                             line=dict(color='#2196F3')))
    fig.add_trace(go.Scatter(x=fut['year'], y=fut['predicted_loss_ha'],
                             mode='lines+markers', name='Forecast',
                             line=dict(color='#F44336', dash='dash')))
    if 'lower_bound' in fut.columns:
        fig.add_trace(go.Scatter(x=fut['year'], y=fut['upper_bound'],
                                 mode='lines', line=dict(width=0), showlegend=False))
        fig.add_trace(go.Scatter(x=fut['year'], y=fut['lower_bound'],
                                 mode='lines', fill='tonexty',
                                 fillcolor='rgba(244,67,54,0.15)',
                                 line=dict(width=0), name='95% CI'))
    fig.update_layout(title=f'{region} — Historical & Forecast',
                      xaxis_title='Year', yaxis_title='Tree-Cover Loss (ha)',
                      template='plotly_white', height=380)
    fig.show()


### 8.2  Forecast Summary Table


In [30]:
fs = future_only.groupby('region')['predicted_loss_ha'].agg(['mean','sum']).round(0)
fs.columns = ['Avg Annual Forecast (ha)', 'Total 5-Year Forecast (ha)']
display(fs.sort_values('Avg Annual Forecast (ha)', ascending=False))


,Avg Annual Forecast (ha),Total 5-Year Forecast (ha)
region,,
Pará,"807,272.00",4036360
Mato Grosso,"546,200.00",2731000
Amazonas,"395,727.00",1978635
Maranhão,"358,542.00",1792710
Rondônia,"219,627.00",1098135
Bahia,"168,063.00",840315
Acre,"141,103.00",705515
Tocantins,"105,202.00",526010
Piauí,"98,383.00",491915


---
<a id="risk"></a>
## 9 · Risk Assessment

| Indicator | Weight | Description |
|-----------|--------|-------------|
| Recent loss rate | 30 % | Avg annual loss rate, last 3 years |
| Loss trend | 25 % | Linear slope, last 3 years |
| Cumulative loss % | 25 % | Total loss / baseline extent |
| Predicted future loss | 20 % | Model-predicted avg future loss |

> ⚠️ Analytical exercise for academic purposes — NOT an official conservation classification.


In [31]:
# Build predicted_future dict for risk assessment
pred_future = future_only.groupby('region')['predicted_loss_ha'].mean().to_dict()
risk_df = full_risk_assessment(processed, predicted_future=pred_future)

print("DEFORESTATION RISK ASSESSMENT")
print("=" * 60)
display(risk_df.sort_values('risk_score', ascending=False)[
    ['region', 'risk_score', 'risk_category']
].reset_index(drop=True))
print(f"\nDistribution: {risk_df['risk_category'].value_counts().to_dict()}")


2026-11-21 20:44:15,277 - deforestation_ml.risk_assessment - INFO - Risk assessment complete: 27 regions. Distribution: {'High': 7, 'Low': 7, 'Very High': 7, 'Moderate': 6}


DEFORESTATION RISK ASSESSMENT


,region,risk_score,risk_category
0,Maranhão,0.83,Very High
1,Rondônia,0.70,Very High
2,Pará,0.62,Very High
3,Mato Grosso,0.56,Very High
4,Bahia,0.54,Very High
5,Tocantins,0.53,Very High
6,Mato Grosso do Sul,0.50,Very High
7,Sergipe,0.47,High
8,Paraná,0.47,High
9,Acre,0.47,High



Distribution: {'High': 7, 'Low': 7, 'Very High': 7, 'Moderate': 6}


In [32]:
color_map = {'Very High': '#d32f2f', 'High': '#f57c00', 'Moderate': '#fbc02d', 'Low': '#388e3c'}
fig = px.bar(risk_df.sort_values('risk_score', ascending=True),
             x='risk_score', y='region', color='risk_category', orientation='h',
             title='Deforestation Risk Ranking by Region',
             labels={'risk_score': 'Risk Score (0–1)', 'region': ''},
             color_discrete_map=color_map)
fig.update_layout(template='plotly_white', height=700)
fig.show()


In [33]:
dist = risk_df['risk_category'].value_counts()
fig = px.pie(values=dist.values, names=dist.index,
             title='Risk Category Distribution',
             color=dist.index, color_discrete_map=color_map)
fig.update_layout(height=400)
fig.show()


---
<a id="carbon"></a>
## 10 · Carbon Emissions Analysis

Carbon data from the GFW forest-carbon-flux model. Units: Mg CO₂e (1 Mg = 1 metric tonne).


In [34]:
if 'carbon_emissions_Mg_CO2e' in processed.columns:
    ca = processed.groupby('year')['carbon_emissions_Mg_CO2e'].sum().reset_index()
    fig = px.line(ca, x='year', y='carbon_emissions_Mg_CO2e',
                  title='Annual Carbon Emissions from Tree-Cover Loss (Brazil)',
                  labels={'carbon_emissions_Mg_CO2e': 'Emissions (Mg CO₂e)', 'year': 'Year'},
                  markers=True)
    fig.update_layout(template='plotly_white', height=400)
    fig.show()
    tc = processed['carbon_emissions_Mg_CO2e'].sum()
    print(f"Total emissions (2001–2022): {tc:,.0f} Mg CO₂e  ({tc/1e9:.2f} billion tonnes)")
else:
    print("Carbon data not available.")


Total emissions (2001–2022): 36,134,302,097 Mg CO₂e  (36.13 billion tonnes)


In [35]:
if 'carbon_emissions_Mg_CO2e' in processed.columns:
    cr = processed.groupby('region')['carbon_emissions_Mg_CO2e'].sum().sort_values(ascending=True).reset_index()
    fig = px.bar(cr, x='carbon_emissions_Mg_CO2e', y='region', orientation='h',
                 title='Total Carbon Emissions by Region (2001–2022)',
                 labels={'carbon_emissions_Mg_CO2e': 'Emissions (Mg CO₂e)', 'region': ''},
                 color='carbon_emissions_Mg_CO2e', color_continuous_scale='Reds')
    fig.update_layout(template='plotly_white', height=700, showlegend=False)
    fig.show()


In [36]:
if 'carbon_emissions_Mg_CO2e' in processed.columns:
    fig = px.scatter(processed, x='tc_loss_ha', y='carbon_emissions_Mg_CO2e',
                     color='region', hover_data=['year'],
                     title='Tree-Cover Loss vs Carbon Emissions',
                     labels={'tc_loss_ha': 'Loss (ha)', 'carbon_emissions_Mg_CO2e': 'Emissions (Mg CO₂e)'})
    fig.update_layout(template='plotly_white', height=500, showlegend=False)
    fig.show()
    r = processed['tc_loss_ha'].corr(processed['carbon_emissions_Mg_CO2e'])
    print(f"Pearson correlation (loss vs emissions): {r:.4f}")


Pearson correlation (loss vs emissions): 0.9816


---
<a id="summary"></a>
## 11 · Summary & Conclusions


In [40]:
print("=" * 70)
print("PROJECT SUMMARY")
print("=" * 70)
print(f"\n📊 Total tree-cover loss (2001–2022): {total_loss:,.0f} ha  ({total_loss/1e6:.1f} M ha)")
print(f"📅 Peak year: {peak_year}  ({peak_val:,.0f} ha)")
print(f"📈 Regions w/ increasing trends: {len(inc)} of 27")
print(f"🏆 Best model (lowest MAE): {best_model_name}")
print(f"   Test MAE: {test_df.loc[test_df['MAE'].idxmin(), 'MAE']:,.0f} ha")
print(f"   Test R² : {test_df.loc[test_df['MAE'].idxmin(), 'R2']:.4f}")
top_risk = risk_df.sort_values('risk_score', ascending=False).iloc[0]
print(f"⚠️  Highest-risk region: {top_risk['region']}  (score: {top_risk['risk_score']:.3f})")

print("\n" + "=" * 70)
print("MODEL COMPARISON — TEST SET")
print("=" * 70)
display(test_df)


PROJECT SUMMARY

📊 Total tree-cover loss (2001–2022): 66,126,533 ha  (66.1 M ha)
📅 Peak year: 2016  (5,378,844 ha)
📈 Regions w/ increasing trends: 0 of 27
🏆 Best model (lowest MAE): Historical Baseline
   Test MAE: 26,092 ha
   Test R² : 0.8697
⚠️  Highest-risk region: Maranhão  (score: 0.835)

MODEL COMPARISON — TEST SET


,Model,MAE,RMSE,R2,n_samples
0,Historical Baseline,"26,091.77","69,294.80",0.87,81
1,Random Forest,"37,385.88","77,939.03",0.84,81
2,XGBoost,"52,121.05","113,591.60",0.65,81
3,Linear Regression,"53,132.62","101,483.19",0.72,81


### Limitations
- Tree-cover loss ≠ permanent deforestation (includes fire, harvest, natural disturbance)
- 22 annual observations per region limits model complexity
- No causal data (roads, population, commodity prices, policy)
- Forecasts are estimates — cannot predict policy changes or climate events
- Risk categories are analytical, not official classifications
- Feature importance = statistical association, NOT causation

---
**Data Source:** Global Forest Change (Hansen et al., 2013), University of Maryland / GLAD Lab.
All metrics computed from real model evaluations using chronological splits. No data fabricated.
